# Notebook 4 — UCSF RMAC Radiomics Feature Extraction
**Dataset:** UCSF RMAC | **Format:** HDF5 | **Phase:** Arterial | **Processes:** Original + Augmented

---

## What This Notebook Does

This notebook extracts radiomics features from UCSF RMAC HDF5 files using PyRadiomics. It runs on both the original AML images and the three augmented versions produced by Notebook 2 (eroded, dilated, flipped). Each subset is saved as a separate CSV file. This is the final step before ML modeling.

**Pipeline position:**
```
[Notebook 1] → [Notebook 2] → [Notebook 3] → [Notebook 4 — YOU ARE HERE]
 KiTS23 Aug     UCSF Aug       KiTS23 Extr    UCSF Feature Extraction
```

**Inputs:**
- Original UCSF AML HDF5 files
- Augmented UCSF AML HDF5 files from Notebook 2 (eroded / dilated / flipped)
- `pyradiomics_params.yaml` — the same file used in Notebook 3

**Outputs (one CSV per subset):**
```
CSV/ucsf_original_aml.csv
CSV/ucsf_augmented_eroded.csv
CSV/ucsf_augmented_dilated.csv
CSV/ucsf_augmented_flipped.csv
```

---

## Key Fix vs. Original Pipeline — Voxel Spacing (Critical)

The original notebook loaded HDF5 arrays with:
```python
image = sitk.GetImageFromArray(image_array)
# spacing defaults to (1.0, 1.0, 1.0) — WRONG
```

SimpleITK has no way to infer real spacing from a raw numpy array. It defaults to `(1.0, 1.0, 1.0) mm` for every case. The real UCSF scanner spacing is `0.762 x 0.762 x 5.0 mm` and is stored in HDF5 file-level attributes:
```
f.attrs['arterial_pixdim'] → [0.761719, 0.761719, 5.0]
```

This notebook reads real spacing from attributes and sets it on the SimpleITK image before passing to PyRadiomics:
```python
spacing = f.attrs['arterial_pixdim']
image.SetSpacing((float(spacing[0]), float(spacing[1]), float(spacing[2])))
```

After this fix, the YAML resampling step brings both KiTS23 and UCSF to the same 1x1x1 mm grid. Features are then computed on an identical geometric foundation for both datasets.

**Without this fix:** UCSF shape features are in voxel units, not mm. A tumor with real volume 130,000 mm³ would appear as 45,000 (voxel units) — comparable to a KiTS23 tumor of 45,000 mm³. The model learns dataset-origin signals, not tumor biology.

---

## Why the Same YAML as Notebook 3

After spacing is correctly set on the SimpleITK image, the YAML's resampling step brings all UCSF cases to 1x1x1 mm — the same target as KiTS23. Bin width (25 HU) and all other settings are identical. The resulting feature columns are directly comparable and can be safely merged into one dataframe for ML modeling.

---

## Dataset Key Note

Original HDF5 files and Notebook 2 augmented files both store data under `'arterial'` and `'mask'` keys. This notebook reads from those keys consistently. The original pipeline used `'image'` as the key in augmented files — this has been corrected in Notebook 2.

## 1. Install Dependencies

In [ ]:
!pip install SimpleITK nibabel h5py pandas -q
!pip install git+https://github.com/Radiomics/pyradiomics.git -q

## 2. Imports

In [ ]:
import os
import h5py
import pandas as pd
import numpy as np
import SimpleITK as sitk
from radiomics import featureextractor

print("Libraries loaded.")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Initialise Extractor — Same YAML as Notebook 3

In [ ]:
YAML_PATH = "/content/drive/MyDrive/pyradiomics_params.yaml"

extractor = featureextractor.RadiomicsFeatureExtractor(YAML_PATH)
print("Extractor initialised with parameter file:", YAML_PATH)
print("Resampling to:", extractor.settings.get('resampledPixelSpacing'))
print("Bin width     :", extractor.settings.get('binWidth'))

## 5. Paths Configuration

In [ ]:
PHASE = "arterial"

# Original UCSF HDF5 files
ORIGINAL_DIR = "/content/drive/MyDrive/hdf5/ucsf-rmac-dataset/hdf5_files/"

# Augmented directories (output of Notebook 2)
AUG_ROOT    = "/content/drive/MyDrive/hdf5-Augmentation"
AUG_ERODED  = os.path.join(AUG_ROOT, "eroded")
AUG_DILATED = os.path.join(AUG_ROOT, "dilated")
AUG_FLIPPED = os.path.join(AUG_ROOT, "flipped")

# CSV output directory
CSV_OUT = "/content/drive/MyDrive/CSV/ucsf/"
os.makedirs(CSV_OUT, exist_ok=True)

## 6. Feature Extraction Function — HDF5

The critical fix is in `load_hdf5_as_sitk_images`: spacing is read from `f.attrs` and set on the SimpleITK image before passing to PyRadiomics.

In [ ]:
def load_hdf5_as_sitk_images(file_path, phase="arterial"):
    """
    Loads an HDF5 file and returns SimpleITK image and mask objects
    with real voxel spacing set from file attributes.

    FIX vs original pipeline:
    Original used sitk.GetImageFromArray() with no spacing — defaulted
    to (1,1,1) mm. This function reads real spacing from
    f.attrs['{phase}_pixdim'] and sets it before returning.

    Returns:
        sitk_image : SimpleITK.Image with correct spacing
        sitk_mask  : SimpleITK.Image with correct spacing
        pid        : str — patient ID from attrs
    """
    with h5py.File(file_path, "r") as f:
        image_array = f[phase][:]
        mask_array  = f["mask"][:].astype(np.uint8)
        attrs       = dict(f.attrs)

    # Read real spacing — stored as [x_mm, y_mm, z_mm]
    spacing_key = f"{phase}_pixdim"
    if spacing_key in attrs:
        raw_spacing = attrs[spacing_key]
        spacing = (float(raw_spacing[0]), float(raw_spacing[1]), float(raw_spacing[2]))
    else:
        print(f"  WARNING: '{spacing_key}' not found in {os.path.basename(file_path)} "
              f"— spacing defaulting to (1,1,1). Check Notebook 2 output.")
        spacing = (1.0, 1.0, 1.0)

    # Create SimpleITK images and set real spacing
    sitk_image = sitk.GetImageFromArray(image_array)
    sitk_image.SetSpacing(spacing)

    sitk_mask  = sitk.GetImageFromArray(mask_array)
    sitk_mask.SetSpacing(spacing)

    pid = attrs.get("PID", os.path.basename(file_path).replace(".hdf5", ""))

    return sitk_image, sitk_mask, str(pid)


def run_extraction_on_hdf5_directory(hdf5_dir, extractor, output_csv,
                                     phase="arterial", label=None):
    """
    Runs feature extraction on all HDF5 files in hdf5_dir.
    Saves results to output_csv.

    Parameters:
        hdf5_dir   : str — directory containing .hdf5 files
        extractor  : RadiomicsFeatureExtractor initialised with YAML
        output_csv : str — full path for output CSV
        phase      : str — CT phase key in HDF5 (default 'arterial')
        label      : str or None — label column value
    """
    files     = sorted([f for f in os.listdir(hdf5_dir) if f.endswith(".hdf5")])
    data_rows = []

    print(f"\nExtracting from: {hdf5_dir}")
    print(f"Files found: {len(files)}")

    for fname in files:
        file_path = os.path.join(hdf5_dir, fname)
        print(f"  Processing: {fname}")

        try:
            sitk_image, sitk_mask, pid = load_hdf5_as_sitk_images(file_path, phase)

            features = dict(extractor.execute(sitk_image, sitk_mask))
            features["case_id"] = pid
            if label is not None:
                features["label"] = label
            data_rows.append(features)

        except Exception as e:
            print(f"  ERROR in {fname}: {e}")
            continue

    if data_rows:
        df = pd.DataFrame(data_rows)
        df.to_csv(output_csv, index=False)
        print(f"  Saved {len(df)} rows → {output_csv}")
    else:
        print(f"  No valid files extracted.")

    return data_rows

## 7. Run — Original UCSF AML Cases

In [ ]:
run_extraction_on_hdf5_directory(
    hdf5_dir   = ORIGINAL_DIR,
    extractor  = extractor,
    output_csv = os.path.join(CSV_OUT, "ucsf_original_aml.csv"),
    phase      = PHASE,
    label      = "AML"
)

## 8. Run — Augmented UCSF AML Cases (Eroded / Dilated / Flipped)

In [ ]:
for aug_name, aug_dir in [("eroded",  AUG_ERODED),
                           ("dilated", AUG_DILATED),
                           ("flipped", AUG_FLIPPED)]:
    run_extraction_on_hdf5_directory(
        hdf5_dir   = aug_dir,
        extractor  = extractor,
        output_csv = os.path.join(CSV_OUT, f"ucsf_augmented_{aug_name}.csv"),
        phase      = PHASE,
        label      = "AML"
    )

## 9. Verify Output — Spacing and Column Consistency with KiTS23

In [ ]:
# Summary of all output CSVs
csv_files = sorted([f for f in os.listdir(CSV_OUT) if f.endswith(".csv")])
print(f"{'File':<45} {'Rows':>6} {'Cols':>6}")
print("-" * 60)
for fname in csv_files:
    df = pd.read_csv(os.path.join(CSV_OUT, fname))
    print(f"{fname:<45} {len(df):>6} {len(df.columns):>6}")

# Confirm spacing resampled to 1x1x1
print("\n--- Spacing after resampling (should be 1.0, 1.0, 1.0) ---")
sample_csv = os.path.join(CSV_OUT, "ucsf_original_aml.csv")
if os.path.exists(sample_csv):
    df = pd.read_csv(sample_csv)
    spacing_col = [c for c in df.columns if "Spacing" in c]
    if spacing_col:
        print(df[spacing_col[0]].unique())

# Cross-check column consistency with KiTS23 output
print("\n--- Column consistency check vs KiTS23 ---")
kits_csv = "/content/drive/MyDrive/CSV/kits23/kits23_original_aml.csv"
ucsf_csv = os.path.join(CSV_OUT, "ucsf_original_aml.csv")
if os.path.exists(kits_csv) and os.path.exists(ucsf_csv):
    kits_cols = set(pd.read_csv(kits_csv).columns)
    ucsf_cols = set(pd.read_csv(ucsf_csv).columns)
    only_kits = kits_cols - ucsf_cols
    only_ucsf = ucsf_cols - kits_cols
    shared    = kits_cols & ucsf_cols
    print(f"Shared columns    : {len(shared)}")
    print(f"Only in KiTS23    : {sorted(only_kits)}")
    print(f"Only in UCSF      : {sorted(only_ucsf)}")
    if not only_kits and not only_ucsf:
        print("All columns match — datasets are ready to merge.")
else:
    print("One or both CSVs not found — run Notebook 3 first.")